# Importando bibliotecas necessárias


In [20]:
import pandas as pd #biblioteca pandas
import numpy as np #biblioteca numpy
import plotly.express as px # para fazer gráficos
from sklearn.compose import make_column_transformer #para transformar colunas de dados em outros formatos
from sklearn.preprocessing import OneHotEncoder # para converter colunas não numéricas
from imblearn.over_sampling import SMOTE # para balancear dados de evasão
from sklearn.preprocessing import StandardScaler # para padronizar e normalizar dados

# Extração do arquivo tratado

In [5]:
## carregando arquivo TelecomX_Dados_Tratados.csv

df = pd.read_csv('TelecomX_Dados_Tratados.csv')

df

,churn,customer_gender,customer_seniorcitizen,customer_partner,customer_dependents,customer_tenure,phone_phoneservice,phone_multiplelines,internet_internetservice,internet_onlinesecurity,internet_onlinebackup,internet_deviceprotection,internet_techsupport,internet_streamingtv,internet_streamingmovies,account_contract,account_paperlessbilling,account_paymentmethod,account_charges_monthly,account_charges_total
0,0,feminino,0,1,1,9,1,0,dsl,0,1,0,1,1,0,anual,1,cheque por correio,65.60,593.30
1,0,masculino,0,0,0,9,1,1,dsl,0,0,0,0,0,1,mensal,0,cheque por correio,59.90,542.40
2,1,masculino,0,0,0,4,1,0,fibra ótica,0,0,1,0,0,0,mensal,1,cheque eletrônico,73.90,280.85
3,1,masculino,1,1,0,13,1,0,fibra ótica,0,1,1,0,1,1,mensal,1,cheque eletrônico,98.00,1237.85
4,1,feminino,1,1,0,3,1,0,fibra ótica,0,0,0,1,1,0,mensal,1,cheque por correio,83.90,267.40
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7038,0,feminino,0,0,0,13,1,0,dsl,1,0,0,1,0,0,anual,0,cheque por correio,55.15,742.90
7039,1,masculino,0,1,0,22,1,1,fibra ótica,0,0,0,0,0,1,mensal,1,cheque eletrônico,85.10,1873.70
7040,0,masculino,0,0,0,2,1,0,dsl,0,1,0,0,0,0,mensal,1,cheque por correio,50.30,92.75
7041,0,masculino,0,1,1,67,1,0,dsl,1,0,1,1,0,1,dois anos,0,cheque por correio,67.85,4627.65


# Encoding

In [10]:
colunas = df.columns
colunas

Index(['churn', 'customer_seniorcitizen', 'customer_partner',
       'customer_dependents', 'customer_tenure', 'phone_phoneservice',
       'phone_multiplelines', 'internet_onlinesecurity',
       'internet_onlinebackup', 'internet_deviceprotection',
       'internet_techsupport', 'internet_streamingtv',
       'internet_streamingmovies', 'account_paperlessbilling',
       'account_charges_monthly', 'account_charges_total',
       'customer_gender_masculino', 'internet_internetservice_dsl',
       'internet_internetservice_fibra ótica', 'account_contract_dois anos',
       'account_contract_mensal', 'account_paymentmethod_cheque eletrônico',
       'account_paymentmethod_cheque por correio',
       'account_paymentmethod_transferência bancária (automática)'],
      dtype='object')

In [14]:
colunas_categoricas = df.select_dtypes(include=['object']).columns.tolist()
print(f"Colunas que serão codificadas: {colunas_categoricas}\n")

colunas = df.columns
colunas


# OneHotEncoder
# drop='first' evita a armadilha da variável dummy (multicolinearidade)
# sparse_output=False garante que o resultado seja um array normal, fácil de unir ao Pandas
encoder = OneHotEncoder(drop='first', sparse_output= False)

# 3. Ajustar (fit) e transformar (transform) os dados categóricos
dados_codificados = encoder.fit_transform(df[colunas_categoricas])

# 4. Resgatar os nomes exatos das novas colunas geradas pelo encoder
nomes_novas_colunas = encoder.get_feature_names_out(colunas_categoricas)

# 5. Transformar o array gerado de volta em um DataFrame do Pandas
df_codificado = pd.DataFrame(dados_codificados, columns=nomes_novas_colunas, index=df.index)

# 6. Unir (concatenar) o novo DataFrame codificado ao original e remover as colunas de texto antigas
df = pd.concat([df.drop(columns=colunas_categoricas), df_codificado], axis=1)

df.head()

Colunas que serão codificadas: []



,churn,customer_seniorcitizen,customer_partner,customer_dependents,customer_tenure,phone_phoneservice,phone_multiplelines,internet_onlinesecurity,internet_onlinebackup,internet_deviceprotection,...,account_charges_monthly,account_charges_total,customer_gender_masculino,internet_internetservice_dsl,internet_internetservice_fibra ótica,account_contract_dois anos,account_contract_mensal,account_paymentmethod_cheque eletrônico,account_paymentmethod_cheque por correio,account_paymentmethod_transferência bancária (automática)
0,0,0,1,1,9,1,0,0,1,0,...,65.6,593.30,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0,0,0,0,9,1,1,0,0,0,...,59.9,542.40,1.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
2,1,0,0,0,4,1,0,0,0,1,...,73.9,280.85,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
3,1,1,1,0,13,1,0,0,1,1,...,98.0,1237.85,1.0,0.0,1.0,0.0,1.0,1.0,0.0,0.0
4,1,1,1,0,3,1,0,0,0,0,...,83.9,267.40,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0


# Verificar taxa de evasão dos clientes

In [16]:
# 1. Calcular a contagem absoluta de cada classe
contagem_absoluta = df['churn'].value_counts()
print("Contagem absoluta de clientes:")
print(contagem_absoluta)

# 2. Calcular a proporção (porcentagem) de cada classe
proporcao = df['churn'].value_counts(normalize=True) * 100
print("\nProporção das classes (%):")
print(proporcao.round(2)) # round(2) para deixar com duas casas decimais

Contagem absoluta de clientes:
churn
0    5174
1    1869
Name: count, dtype: int64

Proporção das classes (%):
churn
0    73.46
1    26.54
Name: proportion, dtype: float64


# Balanceando dados utilizando SMOTE

In [18]:
# 1. Separar as variáveis explicativas (X) da variável alvo (y)
X = df.drop(columns=['churn'])
y = df['churn']

# 2. Instanciar o algoritmo SMOTE
# O random_state=42 garante que o resultado será o mesmo toda vez que você rodar
smote = SMOTE(random_state=42)

# 3. Aplicar o balanceamento aos dados
X_balanceado, y_balanceado = smote.fit_resample(X, y)

# 4. Verificar o resultado antes e depois
print("Contagem de classes ANTES do SMOTE:")
print(y.value_counts())

print("\nContagem de classes APÓS o SMOTE:")
print(y_balanceado.value_counts())

Contagem de classes ANTES do SMOTE:
churn
0    5174
1    1869
Name: count, dtype: int64

Contagem de classes APÓS o SMOTE:
churn
0    5174
1    5174
Name: count, dtype: int64


# Normalização e Padronização

In [21]:


# 1. Definir quais são as colunas numéricas contínuas
colunas_continuas = ['customer_tenure', 'account_charges_monthly', 'account_charges_total']

# 2. Fazer uma cópia do X_balanceado para manter os dados organizados
X_padronizado = X_balanceado.copy()

# 3. Instanciar o padronizador
scaler = StandardScaler()

# 4. Ajustar e transformar apenas as colunas contínuas
X_padronizado[colunas_continuas] = scaler.fit_transform(X_padronizado[colunas_continuas])

# 5. Visualizar como ficaram os dados após a transformação
print("Dados numéricos após a padronização (média ~ 0, desvio padrão ~ 1):")
display(X_padronizado[colunas_continuas].head())

Dados numéricos após a padronização (média ~ 0, desvio padrão ~ 1):


,customer_tenure,account_charges_monthly,account_charges_total
0,-0.777744,-0.084753,-0.664636
1,-0.777744,-0.283258,-0.687910
2,-0.986192,0.204298,-0.807506
3,-0.610986,1.043591,-0.369911
4,-1.027882,0.552552,-0.813656


# Análise de correlação